### Data extraction

In [ ]:
# Clone the GitHub repository and load the data
!git clone https://git.wur.nl/dijk097/ml2023_projectmbf.git data

# Import necessary libraries
import pandas as pd

# Load the dataset
data = pd.read_csv('data/data_labeled_fixed.csv')

# Remove "-" from column names
data.columns = data.columns.str.replace("-", "")

# Create a DataFrame for Week 1 (gene expression values and cell class label)
data_week1 = data.drop(data.columns[-2], axis=1)

# Display the first few rows of the DataFrame
data_week1.head()

Cloning into 'data'...
remote: Enumerating objects: 38, done.
remote: Total 38 (delta 0), reused 0 (delta 0), pack-reused 38
Receiving objects: 100% (38/38), 6.73 MiB | 3.12 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Updating files: 100% (12/12), done.


,Acin1,Actb,Agap1,Ahi1,Akap11,Akap9,Aldoa,Aldoc,Ankrd12,Anp32a,...,Zranb2,mtCo1,mtCytb,mtNd1,mtNd2,mtNd4,mtNd5,mtRnr1,mtRnr2,CLASS
0,3,0,0,1,2,0,1,0,3,2,...,0,0,12,3,5,5,4,2,25,Cone Bipolar ON
1,0,1,0,0,0,0,3,0,0,4,...,0,1,5,1,0,2,1,0,16,Cone Bipolar OFF
2,0,0,0,0,0,0,2,1,0,0,...,2,0,12,5,3,10,4,1,14,Cone Bipolar ON
3,0,1,0,4,0,0,1,0,0,1,...,0,0,8,1,3,3,1,0,7,Cone Bipolar OFF
4,0,1,0,1,0,0,7,1,2,4,...,0,0,17,7,3,0,4,0,3,Cone Bipolar ON


In [ ]:
# Import necessary libraries
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

## **Regression**

Model 1 : Calm1_vs_all_cells

In [ ]:
# Model for Calm1 using all cells
model_name = 'Calm1_Model_AllCells'

X = data_week1.drop(columns=['CLASS', 'Calm1'])
y = data_week1['Calm1']

train, test = train_test_split(data_week1, random_state=42)
formula = 'Calm1 ~ ' + ' + '.join(X.columns)
model = smf.ols(formula=formula, data=train).fit()

print(f"\n{model_name} Summary:")
print(model.summary())

predictions = model.predict(test)
mse = mean_squared_error(test['Calm1'], predictions)
r2 = r2_score(test['Calm1'], predictions)
print(f"Results for {model_name} - MSE: {mse}, R2: {r2}")



Calm1_Model_AllCells Summary:
                            OLS Regression Results                            
Dep. Variable:                  Calm1   R-squared:                       0.805
Model:                            OLS   Adj. R-squared:                  0.768
Method:                 Least Squares   F-statistic:                     21.47
Date:                Sun, 03 Mar 2024   Prob (F-statistic):               0.00
Time:                        20:12:55   Log-Likelihood:                -8058.0
No. Observations:                2793   AIC:                         1.702e+04
Df Residuals:                    2341   BIC:                         1.970e+04
Df Model:                         451                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept      

In [ ]:
model = smf.ols(formula=formula, data=train).fit()

# Get the coefficients from the model summary
coefficients = model.params

sorted_coefficients = coefficients.sort_values(ascending=False)

# Extract the names of the most important features
most_important_features = sorted_coefficients.index

# Display the most important features
print("Most Important Features:")
print(most_important_features[:9])


Most Important Features:
Index(['Fezf1', 'Car8', 'Intercept', 'Prkca', 'Qpct', 'Fam171b', 'Macf1',
       'Lin7a', 'Cck'],
      dtype='object')


In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X_with_constant = add_constant(X)

# Calculate VIF
vif_data = pd.DataFrame({
    "VIF": [variance_inflation_factor(X_with_constant.values, i) for i in range(X_with_constant.shape[1])],
    "feature": X_with_constant.columns
})

# Summarize the VIF values
vif_above_5_below_10 = vif_data[(vif_data['VIF'] > 5) & (vif_data['VIF'] < 10)].shape[0]
vif_above_10 = vif_data[vif_data['VIF'] > 10].shape[0]

print(f"Number of predictors with VIF > 5 and <10: {vif_above_5_below_10}") #moderate level 190
print(f"Number of predictors with VIF > 10: {vif_above_10}") #more severe level 256

#We would conclude that 446 out of 451 of our predictors are highly correlated with each other(indicating multicollinearity)
#and this might cause a problem in our multiple linear regression

Number of predictors with VIF > 5 and <10: 0
Number of predictors with VIF > 10: 1


Model 2 : Calm1_ Rod_Biplolar

In [ ]:
# Model for Calm1 using Rod Bipolar cells
model_name = 'Calm1_Model_RodBipolar'
filtered_data = data_week1[data_week1['CLASS'] == 'Rod Bipolar'].copy()
X = filtered_data.drop(columns=['CLASS', 'Calm1'])
y = filtered_data['Calm1']

train, test = train_test_split(filtered_data, random_state=42)
formula = 'Calm1 ~ ' + ' + '.join(X.columns)
model_1 = smf.ols(formula=formula, data=train).fit()
print(f"\n{model_name} Summary:")
print(model_1.summary())

predictions = model_1.predict(test)
mse = mean_squared_error(test['Calm1'], predictions)
r2 = r2_score(test['Calm1'], predictions)
print(f"Results for {model_name} - MSE: {mse}, R2: {r2}")



Calm1_Model_RodBipolar Summary:


/usr/local/lib/python3.10/dist-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: divide by zero encountered in divide
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: invalid value encountered in scalar multiply
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/linear_model.py:1717: RuntimeWarning: divide by zero encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid
/usr/local/lib/python3.10/dist-packages/statsmodels/base/model.py:1529: RuntimeWarning: invalid value encountered in multiply
  cov_p = self.normalized_cov_params * scale


                            OLS Regression Results                            
Dep. Variable:                  Calm1   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Sun, 03 Mar 2024   Prob (F-statistic):                nan
Time:                        20:17:19   Log-Likelihood:                 10705.
No. Observations:                 375   AIC:                        -2.066e+04
Df Residuals:                       0   BIC:                        -1.919e+04
Df Model:                         374                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -0.0127        inf         -0

VIF

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X_with_constant = add_constant(X)

# Calculate VIF
vif_data = pd.DataFrame({
    "VIF": [variance_inflation_factor(X_with_constant.values, i) for i in range(X_with_constant.shape[1])],
    "feature": X_with_constant.columns
})

# Summarize the VIF values
vif_above_5_below_10 = vif_data[(vif_data['VIF'] > 5) & (vif_data['VIF'] < 10)].shape[0]
vif_above_10 = vif_data[vif_data['VIF'] > 10].shape[0]

print(f"Number of predictors with VIF > 5 and <10: {vif_above_5_below_10}") #moderate level 190
print(f"Number of predictors with VIF > 10: {vif_above_10}") #more severe level 256

#We would conclude that 446 out of 451 of our predictors are highly correlated with each other(indicating multicollinearity)
#and this might cause a problem in our multiple linear regression

/usr/local/lib/python3.10/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


Number of predictors with VIF > 5 and <10: 190
Number of predictors with VIF > 10: 256


Model 3 : Malat1_all_Cells

In [ ]:
# Model for Malat1 using all cells
model_name = 'Malat1_Model_AllCells'

X = data_week1.drop(columns=['CLASS', 'Malat1'])
y = data_week1['Malat1']

train, test = train_test_split(data_week1, random_state=42)
formula = 'Malat1 ~ ' + ' + '.join(X.columns)
model_3 = smf.ols(formula=formula, data=train).fit()
print(f"\n{model_name} Summary:")
print(model_3.summary())

predictions = model_3.predict(test)
mse = mean_squared_error(test['Malat1'], predictions)
r2 = r2_score(test['Malat1'], predictions)
print(f"Results for {model_name} - MSE: {mse}, R2: {r2}")



Malat1_Model_AllCells Summary:
                            OLS Regression Results                            
Dep. Variable:                 Malat1   R-squared:                       0.565
Model:                            OLS   Adj. R-squared:                  0.481
Method:                 Least Squares   F-statistic:                     6.732
Date:                Sun, 03 Mar 2024   Prob (F-statistic):          2.41e-213
Time:                        20:18:44   Log-Likelihood:                -10325.
No. Observations:                2793   AIC:                         2.155e+04
Df Residuals:                    2341   BIC:                         2.424e+04
Df Model:                         451                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept     

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X_with_constant = add_constant(X)

# Calculate VIF
vif_data = pd.DataFrame({
    "VIF": [variance_inflation_factor(X_with_constant.values, i) for i in range(X_with_constant.shape[1])],
    "feature": X_with_constant.columns
})

# Summarize the VIF values
vif_above_5_below_10 = vif_data[(vif_data['VIF'] > 5) & (vif_data['VIF'] < 10)].shape[0]
vif_above_10 = vif_data[vif_data['VIF'] > 10].shape[0]

print(f"Number of predictors with VIF > 5 and <10: {vif_above_5_below_10}") #moderate level 190
print(f"Number of predictors with VIF > 10: {vif_above_10}") #more severe level 256

#We would conclude that 446 out of 451 of our predictors are highly correlated with each other(indicating multicollinearity)
#and this might cause a problem in our multiple linear regression

Number of predictors with VIF > 5 and <10: 0
Number of predictors with VIF > 10: 1


Model 4 : Malat1_Amacrine_cells

In [ ]:
# Model for Malat1 using Amacrine cells
model_name = 'Malat1_Model_Amacrine'
filtered_data = data_week1[data_week1['CLASS'] == 'Amacrine'].copy()
X = filtered_data.drop(columns=['CLASS', 'Malat1'])
y = filtered_data['Malat1']

train, test = train_test_split(filtered_data, random_state=42)
formula = 'Malat1 ~ ' + ' + '.join(X.columns)
model = smf.ols(formula=formula, data=train).fit()
print(f"\n{model_name} Summary:")
print(model.summary())

predictions = model.predict(test)
mse = mean_squared_error(test['Malat1'], predictions)
r2 = r2_score(test['Malat1'], predictions)
print(f"Results for {model_name} - MSE: {mse}, R2: {r2}")



Malat1_Model_Amacrine Summary:


/usr/local/lib/python3.10/dist-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: divide by zero encountered in divide
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/linear_model.py:1795: RuntimeWarning: invalid value encountered in scalar multiply
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/linear_model.py:1717: RuntimeWarning: divide by zero encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid
/usr/local/lib/python3.10/dist-packages/statsmodels/base/model.py:1529: RuntimeWarning: invalid value encountered in multiply
  cov_p = self.normalized_cov_params * scale


                            OLS Regression Results                            
Dep. Variable:                 Malat1   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Sun, 03 Mar 2024   Prob (F-statistic):                nan
Time:                        20:23:05   Log-Likelihood:                 2778.3
No. Observations:                  96   AIC:                            -5365.
Df Residuals:                       0   BIC:                            -5118.
Df Model:                          95                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         0.3968        inf          0

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X_with_constant = add_constant(X)

# Calculate VIF
vif_data = pd.DataFrame({
    "VIF": [variance_inflation_factor(X_with_constant.values, i) for i in range(X_with_constant.shape[1])],
    "feature": X_with_constant.columns
})

# Summarize the VIF values
vif_above_5_below_10 = vif_data[(vif_data['VIF'] > 5) & (vif_data['VIF'] < 10)].shape[0]
vif_above_10 = vif_data[vif_data['VIF'] > 10].shape[0]

print(f"Number of predictors with VIF > 5 and <10: {vif_above_5_below_10}") #moderate level 190
print(f"Number of predictors with VIF > 10: {vif_above_10}") #more severe level 256

#We would conclude that 446 out of 451 of our predictors are highly correlated with each other(indicating multicollinearity)
#and this might cause a problem in our multiple linear regression

/usr/local/lib/python3.10/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/usr/local/lib/python3.10/dist-packages/statsmodels/stats/outliers_influence.py:198: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


Number of predictors with VIF > 5 and <10: 0
Number of predictors with VIF > 10: 445


**Chosen Model: Malat1_Model_AllCells**

b) How many variables are significant and what does this mean?


In [ ]:
# Extract the coefficients and p-values from the summary
coefficients = model_3.params
p_values = model_3.pvalues

# Count the number of significant variables (p-value < 0.05)
significant_variables = (p_values < 0.05).sum()

print(f"\nNumber of Significant Variables: {significant_variables}")
print("Significance level (alpha) is typically set to 0.05.")
print("A variable with a p-value less than alpha is considered significant.")


Number of Significant Variables: 62
Significance level (alpha) is typically set to 0.05.
A variable with a p-value less than alpha is considered significant.


c)

In [ ]:
# Display some example parameter values and interpretations
print("\nExample Parameter Values:")
for variable, coefficient, p_value in zip(coefficients.index, coefficients.values, p_values):
    print(f"{variable}: Coefficient = {coefficient:.4f}, p-value = {p_value:.4f}")



Example Parameter Values:
Intercept: Coefficient = 1.5175, p-value = 0.1122
Acin1: Coefficient = -0.0845, p-value = 0.7084
Actb: Coefficient = -0.1098, p-value = 0.6047
Agap1: Coefficient = -0.1498, p-value = 0.6371
Ahi1: Coefficient = -0.0674, p-value = 0.7890
Akap11: Coefficient = 0.8138, p-value = 0.0065
Akap9: Coefficient = 0.8080, p-value = 0.0053
Aldoa: Coefficient = 0.0425, p-value = 0.8096
Aldoc: Coefficient = -0.5969, p-value = 0.0570
Ankrd12: Coefficient = 0.3218, p-value = 0.0682
Anp32a: Coefficient = -0.0511, p-value = 0.6881
Anp32e: Coefficient = 0.1996, p-value = 0.3058
Aplp1: Coefficient = -0.2109, p-value = 0.5179
Aplp2: Coefficient = -0.0183, p-value = 0.8819
Apoe: Coefficient = 1.3517, p-value = 0.0013
App: Coefficient = 0.3299, p-value = 0.0754
Arglu1: Coefficient = 0.2174, p-value = 0.3801
Arl3: Coefficient = 0.5122, p-value = 0.0992
Arl6ip1: Coefficient = 0.1851, p-value = 0.3236
Arpc2: Coefficient = -0.3329, p-value = 0.2573
Atp1a3: Coefficient = 0.0795, p-value 

# **Classification**

Models:
1. Cone Bipolar OFF vs. the rest
2. Cone Bipolar ON vs. the rest
3. Amacrine vs. the rest
4. Rod Bipolar vs. the rest


## **Logistic** **Regression and K-Nearest Neighbour Classification**

1. **Cone Bipolar OFF vs. the rest**:

In [ ]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Select genes with at least a reasonably high expression value
x = data_week1.drop(["CLASS"], axis=1)
selected_genes = x.columns[x.mean() > 2]
x_selected = x[selected_genes]

# Prepare the target variable (binary encoding for logistic regression)
y = (data_week1['CLASS'] == 'Cone Bipolar OFF').astype(int)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(x_selected, y, test_size=0.25, random_state=42)
# Scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = sm.add_constant (X_train_scaled)
X_test_scaled = sm.add_constant (X_test_scaled)

# Logistic Regression using statsmodels GLM
logistic_model = sm.GLM(y_train, X_train_scaled, family=sm.families.Binomial())
logistic_results = logistic_model.fit()

# Print the summary
print("\nLogistic Regression Summary:")
print(logistic_results.summary())

# Make predictions on the test set
y_pred_logreg_cone_bipolar_off = (logistic_results.predict(X_test_scaled)> 0.5).astype(int)

# Evaluate the model
accuracy_logreg_cone_bipolar_off = accuracy_score(y_test, y_pred_logreg_cone_bipolar_off)
classification_rep_logreg_cone_bipolar_off = classification_report(y_test, y_pred_logreg_cone_bipolar_off)

# Display results
#print(x_selected.shape)
print(f"Accuracy: {accuracy_logreg_cone_bipolar_off:.4f}")
print("Classification Report:")
print(classification_rep_logreg_cone_bipolar_off)



Logistic Regression Summary:
                 Generalized Linear Model Regression Results                  
Dep. Variable:                  CLASS   No. Observations:                 2793
Model:                            GLM   Df Residuals:                     2770
Model Family:                Binomial   Df Model:                           22
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -612.99
Date:                Sun, 03 Mar 2024   Deviance:                       1226.0
Time:                        20:23:18   Pearson chi2:                 8.17e+03
No. Iterations:                     8   Pseudo R-squ. (CS):             0.6076
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.0658 

In [ ]:
#glm_result is the result of the GLM fit from statsmodels
coefficients = logistic_results.params

# Ignoring the intercept
coefficients = coefficients.drop('const', errors='ignore')  # errors='ignore' allows it to proceed if 'const' is not present

# Finding the feature with the highest absolute coefficient value
most_important_feature = coefficients.abs().idxmax()
most_important_value = coefficients[most_important_feature]

print(f"Most important feature: {most_important_feature}, Coefficient Value: {most_important_value}")

Most important feature: x13, Coefficient Value: 2.5774121682750004


In [ ]:
x_selected.columns

Index(['Anp32a', 'Aplp2', 'Atp2b1', 'Calm1', 'Ckb', 'Gng13', 'Hnrnpa2b1',
       'Luc7l3', 'Malat1', 'Meg3', 'Mgarp', 'Pcp2', 'Pcp4', 'Scg2', 'Slc25a4',
       'Syt1', 'Trpm1', 'Ttyh1', 'mtCytb', 'mtNd1', 'mtNd5', 'mtRnr2'],
      dtype='object')

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# K-Nearest Neighbors (KNN)
knn_cone_bipolar_off = KNeighborsClassifier(n_neighbors=3)
knn_cone_bipolar_off.fit(X_train, y_train)
y_pred_knn_cone_bipolar_off = knn_cone_bipolar_off.predict(X_test)

# Evaluate the model
accuracy_knn_cone_bipolar_off = accuracy_score(y_test, y_pred_knn_cone_bipolar_off)
classification_rep_knn_cone_bipolar_off = classification_report(y_test, y_pred_knn_cone_bipolar_off)

# Display results
print("\nK-Nearest Neighbors (KNN) Results for Cone Bipolar OFF vs. Rest:")
print(f"Accuracy: {accuracy_knn_cone_bipolar_off:.4f}")
print("Classification Report:")
print(classification_rep_knn_cone_bipolar_off)



K-Nearest Neighbors (KNN) Results for Cone Bipolar OFF vs. Rest:
Accuracy: 0.8883
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.89      0.90       517
           1       0.87      0.88      0.88       414

    accuracy                           0.89       931
   macro avg       0.89      0.89      0.89       931
weighted avg       0.89      0.89      0.89       931



2. **Cone Bipolar ON vs. Rest**:




In [ ]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Select genes with at least a reasonably high expression value
x = data_week1.drop(["CLASS"], axis=1)
selected_genes = x.columns[x.mean() > 2]
x_selected = x[selected_genes]

# Prepare the target variable (binary encoding for logistic regression)
y_cone_bipolar_on = (data_week1['CLASS'] == 'Cone Bipolar ON').astype(int)

# Split the data into training and test sets
X_train_cone_bipolar_on, X_test_cone_bipolar_on, y_train_cone_bipolar_on, y_test_cone_bipolar_on = train_test_split(x_selected, y_cone_bipolar_on, test_size=0.25, random_state=42)


# Scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_cone_bipolar_on)
X_test_scaled = scaler.transform(X_test_cone_bipolar_on)

X_train_scaled = sm.add_constant (X_train_scaled)
X_test_scaled = sm.add_constant (X_test_scaled)

# Logistic Regression using statsmodels GLM
logistic_model_cone_bipolar_on = sm.GLM(y_train_cone_bipolar_on, X_train_scaled, family=sm.families.Binomial())
logistic_results_cone_bipolar_on = logistic_model_cone_bipolar_on.fit()

# Print the summary
print("\nLogistic Regression Summary for Cone Bipolar ON vs. Rest:")
print(logistic_results_cone_bipolar_on.summary())

# Make predictions on the test set
y_pred_logreg_cone_bipolar_on = (logistic_results_cone_bipolar_on.predict(X_test_scaled) > 0.5).astype(int)

# Evaluate the model
accuracy_logreg_cone_bipolar_on = accuracy_score(y_test_cone_bipolar_on, y_pred_logreg_cone_bipolar_on)
classification_rep_logreg_cone_bipolar_on = classification_report(y_test_cone_bipolar_on, y_pred_logreg_cone_bipolar_on)

# Display results
print(f"Accuracy: {accuracy_logreg_cone_bipolar_on:.4f}")
print("Classification Report:")
print(classification_rep_logreg_cone_bipolar_on)



Logistic Regression Summary for Cone Bipolar ON vs. Rest:
                 Generalized Linear Model Regression Results                  
Dep. Variable:                  CLASS   No. Observations:                 2793
Model:                            GLM   Df Residuals:                     2770
Model Family:                Binomial   Df Model:                           22
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -516.88
Date:                Sun, 03 Mar 2024   Deviance:                       1033.8
Time:                        20:23:18   Pearson chi2:                 1.69e+07
No. Iterations:                     9   Pseudo R-squ. (CS):             0.6176
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------

In [ ]:
#glm_result is the result of the GLM fit from statsmodels
coefficients = logistic_results_cone_bipolar_on.params

# Ignoring the intercept, if you've included it
coefficients = coefficients.drop('const', errors='ignore')  # errors='ignore' allows it to proceed if 'const' is not present

# Finding the feature with the highest absolute coefficient value
most_important_feature = coefficients.abs().idxmax()
most_important_value = coefficients[most_important_feature]

print(f"Most important feature: {most_important_feature}, Coefficient Value: {most_important_value}")

Most important feature: x13, Coefficient Value: -6.618255537422285


In [ ]:
x_selected.columns

Index(['Anp32a', 'Aplp2', 'Atp2b1', 'Calm1', 'Ckb', 'Gng13', 'Hnrnpa2b1',
       'Luc7l3', 'Malat1', 'Meg3', 'Mgarp', 'Pcp2', 'Pcp4', 'Scg2', 'Slc25a4',
       'Syt1', 'Trpm1', 'Ttyh1', 'mtCytb', 'mtNd1', 'mtNd5', 'mtRnr2'],
      dtype='object')

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# K-Nearest Neighbors (KNN)
knn_cone_bipolar_on = KNeighborsClassifier(n_neighbors=3)
knn_cone_bipolar_on.fit(X_train_cone_bipolar_on, y_train_cone_bipolar_on)
y_pred_knn_cone_bipolar_on = knn_cone_bipolar_on.predict(X_test_cone_bipolar_on)

# Evaluate the model
accuracy_knn_cone_bipolar_on = accuracy_score(y_test_cone_bipolar_on, y_pred_knn_cone_bipolar_on)
classification_rep_knn_cone_bipolar_on = classification_report(y_test_cone_bipolar_on, y_pred_knn_cone_bipolar_on)

# Display results
print("\nK-Nearest Neighbors (KNN) Results for Cone Bipolar ON vs. Rest:")
print(f"Accuracy: {accuracy_knn_cone_bipolar_on:.4f}")
print("Classification Report:")
print(classification_rep_knn_cone_bipolar_on)



K-Nearest Neighbors (KNN) Results for Cone Bipolar ON vs. Rest:
Accuracy: 0.9066
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.94      0.92       565
           1       0.90      0.85      0.88       366

    accuracy                           0.91       931
   macro avg       0.91      0.90      0.90       931
weighted avg       0.91      0.91      0.91       931



3. **Amacrine vs Rest**:

In [ ]:
# Select genes with at least a reasonably high expression value
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

x = data_week1.drop(["CLASS"], axis=1)
selected_genes = x.columns[x.mean() > 2]
x_selected = x[selected_genes]

# Prepare the target variable (binary encoding for logistic regression)
y_amacrine = (data_week1['CLASS'] == 'Amacrine').astype(int)

# Split the data into training and test sets
X_train_amacrine, X_test_amacrine, y_train_amacrine, y_test_amacrine = train_test_split(x_selected, y_amacrine, test_size=0.25, random_state=42)

# Scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_amacrine)
X_test_scaled = scaler.transform(X_test_amacrine)

X_train_scaled = sm.add_constant (X_train_scaled)
X_test_scaled = sm.add_constant (X_test_scaled)

# Logistic Regression using statsmodels GLM
logistic_model_amacrine = sm.GLM(y_train_amacrine,X_train_scaled , family=sm.families.Binomial())
logistic_results_amacrine = logistic_model_amacrine.fit()

# Print the summary
print("\nLogistic Regression Summary for Amacrine vs. Rest:")
print(logistic_results_amacrine.summary())

# Make predictions on the test set
y_pred_logreg_amacrine = (logistic_results_amacrine.predict(X_test_scaled) > 0.5).astype(int)

# Evaluate the model
#accuracy_logreg_amacrine = accuracy_score(y_test_amacrine, y_pred_logreg_amacrine)
accuracy_logreg_amacrine = accuracy_score(y_test_amacrine, y_pred_logreg_amacrine)

classification_rep_logreg_amacrine = classification_report(y_test_amacrine, y_pred_logreg_amacrine)

# Display results
print(f"Accuracy: {accuracy_logreg_amacrine:.4f}")
print("Classification Report:")
print(classification_rep_logreg_amacrine)



Logistic Regression Summary for Amacrine vs. Rest:
                 Generalized Linear Model Regression Results                  
Dep. Variable:                  CLASS   No. Observations:                 2793
Model:                            GLM   Df Residuals:                     2770
Model Family:                Binomial   Df Model:                           22
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -115.08
Date:                Sun, 03 Mar 2024   Deviance:                       230.15
Time:                        20:23:19   Pearson chi2:                 1.07e+04
No. Iterations:                    11   Pseudo R-squ. (CS):             0.2044
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------


In [ ]:
#glm_result is the result of the GLM fit from statsmodels
coefficients = logistic_results_amacrine.params

# Ignoring the intercept, if you've included it
coefficients = coefficients.drop('const', errors='ignore')  # errors='ignore' allows it to proceed if 'const' is not present

# Finding the feature with the highest absolute coefficient value
most_important_feature = coefficients.abs().idxmax()
most_important_value = coefficients[most_important_feature]

print(f"Most important feature: {most_important_feature}, Coefficient Value: {most_important_value}")

Most important feature: x6, Coefficient Value: -3.1210489018674594


In [ ]:
x_selected.columns

Index(['Anp32a', 'Aplp2', 'Atp2b1', 'Calm1', 'Ckb', 'Gng13', 'Hnrnpa2b1',
       'Luc7l3', 'Malat1', 'Meg3', 'Mgarp', 'Pcp2', 'Pcp4', 'Scg2', 'Slc25a4',
       'Syt1', 'Trpm1', 'Ttyh1', 'mtCytb', 'mtNd1', 'mtNd5', 'mtRnr2'],
      dtype='object')

In [ ]:
# K-Nearest Neighbors (KNN)
knn_amacrine = KNeighborsClassifier(n_neighbors=3)
knn_amacrine.fit(X_train_amacrine, y_train_amacrine)
y_pred_knn_amacrine = knn_amacrine.predict(X_test_amacrine)

# Evaluate the model
accuracy_knn_amacrine = accuracy_score(y_test_amacrine, y_pred_knn_amacrine)
classification_rep_knn_amacrine = classification_report(y_test_amacrine, y_pred_knn_amacrine)

# Display results
print("\nK-Nearest Neighbors (KNN) Results for Amacrine vs. Rest:")
print(f"Accuracy: {accuracy_knn_amacrine:.4f}")
print("Classification Report:")
print(classification_rep_knn_amacrine)



K-Nearest Neighbors (KNN) Results for Amacrine vs. Rest:
Accuracy: 0.9721
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       903
           1       0.56      0.36      0.43        28

    accuracy                           0.97       931
   macro avg       0.77      0.67      0.71       931
weighted avg       0.97      0.97      0.97       931



4. **Rod Bipolar vs. Rest**

In [ ]:
# Select genes with at least a reasonably high expression value
from sklearn.preprocessing import StandardScaler

x = data_week1.drop(["CLASS"], axis=1)
selected_genes = x.columns[x.mean() > 2]
x_selected = x[selected_genes]

# Prepare the target variable (binary encoding for logistic regression)
y_rod_bipolar = (data_week1['CLASS'] == 'Rod Bipolar').astype(int)

# Split the data into training and test sets
X_train_rod_bipolar, X_test_rod_bipolar, y_train_rod_bipolar, y_test_rod_bipolar = train_test_split(x_selected, y_rod_bipolar, test_size=0.25, random_state=42)

# Scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_rod_bipolar)
X_test_scaled = scaler.transform(X_test_rod_bipolar)

X_train_scaled = sm.add_constant (X_train_scaled)
X_test_scaled = sm.add_constant (X_test_scaled)

# Logistic Regression using statsmodels GLM
logistic_model_rod_bipolar = sm.GLM(y_train_rod_bipolar,X_train_scaled , family=sm.families.Binomial())
logistic_results_rod_bipolar = logistic_model_rod_bipolar.fit()

# Print the summary
print("\nLogistic Regression Summary for Rod Bipolar vs. Rest:")
print(logistic_results_rod_bipolar.summary())

# Make predictions on the test set
#y_pred_logreg_rod_bipolar = (logistic_results_rod_bipolar.predict(X_test_scaled) > 0.5).astype(int)
y_pred_logreg_rod_bipolar = (logistic_results_rod_bipolar.predict(X_test_scaled) > 0.5)


# Evaluate the model
accuracy_logreg_rod_bipolar = accuracy_score(y_test_rod_bipolar, y_pred_logreg_rod_bipolar)
classification_rep_logreg_rod_bipolar = classification_report(y_test_rod_bipolar, y_pred_logreg_rod_bipolar)

# Display results
#print(x_selected.shape)
print(f"Accuracy: {accuracy_logreg_rod_bipolar:.4f}")
print("Classification Report:")
print(classification_rep_logreg_rod_bipolar)



Logistic Regression Summary for Rod Bipolar vs. Rest:
                 Generalized Linear Model Regression Results                  
Dep. Variable:                  CLASS   No. Observations:                 2793
Model:                            GLM   Df Residuals:                     2770
Model Family:                Binomial   Df Model:                           22
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -75.012
Date:                Sun, 03 Mar 2024   Deviance:                       150.02
Time:                        20:23:19   Pearson chi2:                 1.07e+04
No. Iterations:                    10   Pseudo R-squ. (CS):             0.5218
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------

In [ ]:
#glm_result is the result of the GLM fit from statsmodels
coefficients = logistic_results_rod_bipolar.params

# Ignoring the intercept, if you've included it
coefficients = coefficients.drop('const', errors='ignore')  # errors='ignore' allows it to proceed if 'const' is not present

# Finding the feature with the highest absolute coefficient value
most_important_feature = coefficients.abs().idxmax()
most_important_value = coefficients[most_important_feature]

print(f"Most important feature: {most_important_feature}, Coefficient Value: {most_important_value}")

Most important feature: x4, Coefficient Value: 3.0690826587508915


In [ ]:
x_selected.columns

Index(['Anp32a', 'Aplp2', 'Atp2b1', 'Calm1', 'Ckb', 'Gng13', 'Hnrnpa2b1',
       'Luc7l3', 'Malat1', 'Meg3', 'Mgarp', 'Pcp2', 'Pcp4', 'Scg2', 'Slc25a4',
       'Syt1', 'Trpm1', 'Ttyh1', 'mtCytb', 'mtNd1', 'mtNd5', 'mtRnr2'],
      dtype='object')

In [ ]:
# K-Nearest Neighbors (KNN)
knn_rod_bipolar = KNeighborsClassifier(n_neighbors=3)
knn_rod_bipolar.fit(X_train_rod_bipolar, y_train_rod_bipolar)
y_pred_knn_rod_bipolar = knn_rod_bipolar.predict(X_test_rod_bipolar)

# Evaluate the model
accuracy_knn_rod_bipolar = accuracy_score(y_test_rod_bipolar, y_pred_knn_rod_bipolar)
classification_rep_knn_rod_bipolar = classification_report(y_test_rod_bipolar, y_pred_knn_rod_bipolar)

# Display results
print("\nK-Nearest Neighbors (KNN) Results for Rod Bipolar vs. Rest:")
print(f"Accuracy: {accuracy_knn_rod_bipolar:.4f}")
print("Classification Report:")
print(classification_rep_knn_rod_bipolar)



K-Nearest Neighbors (KNN) Results for Rod Bipolar vs. Rest:
Accuracy: 0.9850
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       808
           1       0.92      0.97      0.94       123

    accuracy                           0.98       931
   macro avg       0.96      0.98      0.97       931
weighted avg       0.99      0.98      0.99       931



**a) Analyse the predictive performance**

1. Cone Bipolar OFF vs. Rest:


In [ ]:
# Logistic Regression
print(f"Accuracy (Logistic Regression): {accuracy_logreg_cone_bipolar_off:.4f}")
print("Classification Report:")
print(classification_rep_logreg_cone_bipolar_off)

# K-Nearest Neighbors
print(f"Accuracy (KNN): {accuracy_knn_cone_bipolar_off:.4f}")
print("Classification Report:")
print(classification_rep_knn_cone_bipolar_off)


Accuracy (Logistic Regression): 0.9001
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91       517
           1       0.89      0.88      0.89       414

    accuracy                           0.90       931
   macro avg       0.90      0.90      0.90       931
weighted avg       0.90      0.90      0.90       931

Accuracy (KNN): 0.8883
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.89      0.90       517
           1       0.87      0.88      0.88       414

    accuracy                           0.89       931
   macro avg       0.89      0.89      0.89       931
weighted avg       0.89      0.89      0.89       931



In [ ]:
from sklearn.metrics import confusion_matrix

# Logistic Regression
print(f"Accuracy (Logistic Regression): {accuracy_logreg_cone_bipolar_off:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_logreg_cone_bipolar_off))
print("Classification Report:")
print(classification_rep_logreg_cone_bipolar_off)

# K-Nearest Neighbors
print(f"Accuracy (KNN): {accuracy_knn_cone_bipolar_off:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn_cone_bipolar_off))
print("Classification Report:")
print(classification_rep_knn_cone_bipolar_off)


Accuracy (Logistic Regression): 0.9001
Confusion Matrix:
[[472  45]
 [ 48 366]]
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91       517
           1       0.89      0.88      0.89       414

    accuracy                           0.90       931
   macro avg       0.90      0.90      0.90       931
weighted avg       0.90      0.90      0.90       931

Accuracy (KNN): 0.8883
Confusion Matrix:
[[462  55]
 [ 49 365]]
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.89      0.90       517
           1       0.87      0.88      0.88       414

    accuracy                           0.89       931
   macro avg       0.89      0.89      0.89       931
weighted avg       0.89      0.89      0.89       931



2. Cone Bipolar ON vs. Rest:


In [ ]:
# Logistic Regression
print(f"Accuracy (Logistic Regression): {accuracy_logreg_cone_bipolar_on:.4f}")
print("Classification Report:")
print(classification_rep_logreg_cone_bipolar_on)

# K-Nearest Neighbors
print(f"Accuracy (KNN): {accuracy_knn_cone_bipolar_on:.4f}")
print("Classification Report:")
print(classification_rep_knn_cone_bipolar_on)


Accuracy (Logistic Regression): 0.9205
Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.94      0.93       565
           1       0.90      0.90      0.90       366

    accuracy                           0.92       931
   macro avg       0.92      0.92      0.92       931
weighted avg       0.92      0.92      0.92       931

Accuracy (KNN): 0.9066
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.94      0.92       565
           1       0.90      0.85      0.88       366

    accuracy                           0.91       931
   macro avg       0.91      0.90      0.90       931
weighted avg       0.91      0.91      0.91       931



In [ ]:
from sklearn.metrics import confusion_matrix

# Logistic Regression
print(f"Accuracy (Logistic Regression): {accuracy_logreg_cone_bipolar_on:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test_cone_bipolar_on, y_pred_logreg_cone_bipolar_on))
print("Classification Report:")
print(classification_rep_logreg_cone_bipolar_on)

# K-Nearest Neighbors
print(f"Accuracy (KNN): {accuracy_knn_cone_bipolar_on:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test_cone_bipolar_on, y_pred_knn_cone_bipolar_on))
print("Classification Report:")
print(classification_rep_knn_cone_bipolar_on)


Accuracy (Logistic Regression): 0.9205
Confusion Matrix:
[[529  36]
 [ 38 328]]
Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.94      0.93       565
           1       0.90      0.90      0.90       366

    accuracy                           0.92       931
   macro avg       0.92      0.92      0.92       931
weighted avg       0.92      0.92      0.92       931

Accuracy (KNN): 0.9066
Confusion Matrix:
[[532  33]
 [ 54 312]]
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.94      0.92       565
           1       0.90      0.85      0.88       366

    accuracy                           0.91       931
   macro avg       0.91      0.90      0.90       931
weighted avg       0.91      0.91      0.91       931



3. Amacrine vs. Rest:


In [ ]:
# Logistic Regression
print(f"Accuracy (Logistic Regression): {accuracy_logreg_amacrine:.4f}")
print("Classification Report:")
print(classification_rep_logreg_amacrine)

# K-Nearest Neighbors
print(f"Accuracy (KNN): {accuracy_knn_amacrine:.4f}")
print("Classification Report:")
print(classification_rep_knn_amacrine)


Accuracy (Logistic Regression): 0.9871
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       903
           1       0.79      0.79      0.79        28

    accuracy                           0.99       931
   macro avg       0.89      0.89      0.89       931
weighted avg       0.99      0.99      0.99       931

Accuracy (KNN): 0.9721
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       903
           1       0.56      0.36      0.43        28

    accuracy                           0.97       931
   macro avg       0.77      0.67      0.71       931
weighted avg       0.97      0.97      0.97       931



In [ ]:
from sklearn.metrics import confusion_matrix

# Logistic Regression
print(f"Accuracy (Logistic Regression): {accuracy_logreg_amacrine:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test_amacrine, y_pred_logreg_amacrine))
print("Classification Report:")
print(classification_rep_logreg_amacrine)

# K-Nearest Neighbors
print(f"Accuracy (KNN): {accuracy_knn_amacrine:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test_amacrine, y_pred_knn_amacrine))
print("Classification Report:")
print(classification_rep_knn_amacrine)


Accuracy (Logistic Regression): 0.9871
Confusion Matrix:
[[897   6]
 [  6  22]]
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       903
           1       0.79      0.79      0.79        28

    accuracy                           0.99       931
   macro avg       0.89      0.89      0.89       931
weighted avg       0.99      0.99      0.99       931

Accuracy (KNN): 0.9721
Confusion Matrix:
[[895   8]
 [ 18  10]]
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       903
           1       0.56      0.36      0.43        28

    accuracy                           0.97       931
   macro avg       0.77      0.67      0.71       931
weighted avg       0.97      0.97      0.97       931



4. Rod Bipolar vs. Rest:

In [ ]:
# Logistic Regression
print(f"Accuracy (Logistic Regression): {accuracy_logreg_rod_bipolar:.4f}")
print("Classification Report:")
print(classification_rep_logreg_rod_bipolar)

# K-Nearest Neighbors
print(f"Accuracy (KNN): {accuracy_knn_rod_bipolar:.4f}")
print("Classification Report:")
print(classification_rep_knn_rod_bipolar)


Accuracy (Logistic Regression): 0.9871
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       808
           1       0.93      0.98      0.95       123

    accuracy                           0.99       931
   macro avg       0.96      0.98      0.97       931
weighted avg       0.99      0.99      0.99       931

Accuracy (KNN): 0.9850
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       808
           1       0.92      0.97      0.94       123

    accuracy                           0.98       931
   macro avg       0.96      0.98      0.97       931
weighted avg       0.99      0.98      0.99       931



In [ ]:
from sklearn.metrics import confusion_matrix

# Logistic Regression
print(f"Accuracy (Logistic Regression): {accuracy_logreg_rod_bipolar:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test_rod_bipolar, y_pred_logreg_rod_bipolar))
print("Classification Report:")
print(classification_rep_logreg_rod_bipolar)

# K-Nearest Neighbors
print(f"Accuracy (KNN): {accuracy_knn_rod_bipolar:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test_rod_bipolar, y_pred_knn_rod_bipolar))
print("Classification Report:")
print(classification_rep_knn_rod_bipolar)


Accuracy (Logistic Regression): 0.9871
Confusion Matrix:
[[799   9]
 [  3 120]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       808
           1       0.93      0.98      0.95       123

    accuracy                           0.99       931
   macro avg       0.96      0.98      0.97       931
weighted avg       0.99      0.99      0.99       931

Accuracy (KNN): 0.9850
Confusion Matrix:
[[798  10]
 [  4 119]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       808
           1       0.92      0.97      0.94       123

    accuracy                           0.98       931
   macro avg       0.96      0.98      0.97       931
weighted avg       0.99      0.98      0.99       931



b ) For the KNN model predicting Amacrine vs. the rest, test using a couple of different value of the number of neighbours, and report on how this affects the balance between specificity and sensitivity

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# Test different values of n_neighbors
for n_neighbors in [3, 5, 7, 9]:
    # K-Nearest Neighbors (KNN)
    knn_amacrine = KNeighborsClassifier(n_neighbors=n_neighbors)
    knn_amacrine.fit(X_train_amacrine, y_train_amacrine)
    y_pred_knn_amacrine = knn_amacrine.predict(X_test_amacrine)

    # Evaluate the model
    accuracy_knn_amacrine = accuracy_score(y_test_amacrine, y_pred_knn_amacrine)
    classification_rep_knn_amacrine = classification_report(y_test_amacrine, y_pred_knn_amacrine)

    # Display results for each value of n_neighbors
    print(f"\nK-Nearest Neighbors (KNN) Results for Amacrine vs. Rest (n_neighbors={n_neighbors}):")
    print(f"Accuracy: {accuracy_knn_amacrine:.4f}")
    print("Classification Report:")
    print(classification_rep_knn_amacrine)



K-Nearest Neighbors (KNN) Results for Amacrine vs. Rest (n_neighbors=3):
Accuracy: 0.9721
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       903
           1       0.56      0.36      0.43        28

    accuracy                           0.97       931
   macro avg       0.77      0.67      0.71       931
weighted avg       0.97      0.97      0.97       931


K-Nearest Neighbors (KNN) Results for Amacrine vs. Rest (n_neighbors=5):
Accuracy: 0.9710
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       903
           1       0.55      0.21      0.31        28

    accuracy                           0.97       931
   macro avg       0.76      0.60      0.65       931
weighted avg       0.96      0.97      0.96       931


K-Nearest Neighbors (KNN) Results for Amacrine vs. Rest (n_neighbors=7):
Accuracy: 0.9731
Classification Report:
    

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize variables to keep track of the best k and its accuracy
best_k = 0
best_accuracy = 0

# Test different values of n_neighbors
for n_neighbors in [3, 5, 7, 9]:
    # K-Nearest Neighbors (KNN)
    knn_amacrine = KNeighborsClassifier(n_neighbors=n_neighbors)
    knn_amacrine.fit(X_train_amacrine, y_train_amacrine)
    y_pred_knn_amacrine = knn_amacrine.predict(X_test_amacrine)

    # Evaluate the model
    accuracy_knn_amacrine = accuracy_score(y_test_amacrine, y_pred_knn_amacrine)
    classification_rep_knn_amacrine = classification_report(y_test_amacrine, y_pred_knn_amacrine)

    # Display results for each value of n_neighbors
    print(f"\nK-Nearest Neighbors (KNN) Results for Amacrine vs. Rest (n_neighbors={n_neighbors}):")
    print(f"Accuracy: {accuracy_knn_amacrine:.4f}")
    print("Classification Report:")
    print(classification_rep_knn_amacrine)

    # Check if the current k achieved higher accuracy
    if accuracy_knn_amacrine > best_accuracy:
        best_accuracy = accuracy_knn_amacrine
        best_k = n_neighbors

# Display the best k and its accuracy
print(f"\nBest K: {best_k} with Accuracy: {best_accuracy:.4f}")



K-Nearest Neighbors (KNN) Results for Amacrine vs. Rest (n_neighbors=3):
Accuracy: 0.9721
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       903
           1       0.56      0.36      0.43        28

    accuracy                           0.97       931
   macro avg       0.77      0.67      0.71       931
weighted avg       0.97      0.97      0.97       931


K-Nearest Neighbors (KNN) Results for Amacrine vs. Rest (n_neighbors=5):
Accuracy: 0.9710
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       903
           1       0.55      0.21      0.31        28

    accuracy                           0.97       931
   macro avg       0.76      0.60      0.65       931
weighted avg       0.96      0.97      0.96       931


K-Nearest Neighbors (KNN) Results for Amacrine vs. Rest (n_neighbors=7):
Accuracy: 0.9731
Classification Report:
    

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Initialize variables to keep track of the best k and its accuracy, specificity, and sensitivity
best_k = 0
best_accuracy = 0
best_specificity = 0
best_sensitivity = 0

# Test different values of n_neighbors
for n_neighbors in [1, 3, 5, 7]:
    # K-Nearest Neighbors (KNN)
    knn_amacrine = KNeighborsClassifier(n_neighbors=n_neighbors)
    knn_amacrine.fit(X_train_amacrine, y_train_amacrine)
    y_pred_knn_amacrine = knn_amacrine.predict(X_test_amacrine)

    # Evaluate the model
    accuracy_knn_amacrine = accuracy_score(y_test_amacrine, y_pred_knn_amacrine)
    confusion_mat = confusion_matrix(y_test_amacrine, y_pred_knn_amacrine)

    # Calculate specificity and sensitivity
    true_negative, false_positive, false_negative, true_positive = confusion_mat.ravel()
    specificity = true_negative / (true_negative + false_positive)
    sensitivity = true_positive / (true_positive + false_negative)

    # Display results for each value of n_neighbors
    print(f"\nK-Nearest Neighbors (KNN) Results for Amacrine vs. Rest (n_neighbors={n_neighbors}):")
    print(f"Accuracy: {accuracy_knn_amacrine:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"Sensitivity: {sensitivity:.4f}")
    print("Classification Report:")
    print(classification_report(y_test_amacrine, y_pred_knn_amacrine))

    # Check if the current k achieved higher accuracy
    if accuracy_knn_amacrine > best_accuracy:
        best_accuracy = accuracy_knn_amacrine
        best_k = n_neighbors
        best_specificity = specificity
        best_sensitivity = sensitivity

# Display the best k, accuracy, specificity, and sensitivity
print(f"\nBest K: {best_k} with Accuracy: {best_accuracy:.4f}, Specificity: {best_specificity:.4f}, Sensitivity: {best_sensitivity:.4f}")



K-Nearest Neighbors (KNN) Results for Amacrine vs. Rest (n_neighbors=1):
Accuracy: 0.9603
Specificity: 0.9745
Sensitivity: 0.5000
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.97      0.98       903
           1       0.38      0.50      0.43        28

    accuracy                           0.96       931
   macro avg       0.68      0.74      0.71       931
weighted avg       0.97      0.96      0.96       931


K-Nearest Neighbors (KNN) Results for Amacrine vs. Rest (n_neighbors=3):
Accuracy: 0.9721
Specificity: 0.9911
Sensitivity: 0.3571
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       903
           1       0.56      0.36      0.43        28

    accuracy                           0.97       931
   macro avg       0.77      0.67      0.71       931
weighted avg       0.97      0.97      0.97       931


K-Nearest Neighbors (KNN) Results for